In [1]:
import warnings
for warn in [UserWarning, FutureWarning]: warnings.filterwarnings("ignore", category = warn)

import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import hub
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import torchvision.models as models

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, recall_score, precision_score, balanced_accuracy_score, accuracy_score, classification_report
from sklearn.utils import shuffle

import scipy

from tqdm import tqdm

from datasets import load_dataset, Dataset, Audio
import librosa
from transformers import AutoFeatureExtractor, Wav2Vec2ForSequenceClassification

# from models.swishnet import SwishNet
from models.cnn import CNNModel

from src.utils import AphasiaDatasetMFCC, AphasiaDatasetSpectrogram, AphasiaDatasetWaveform

from collections import Counter

# from torch_audiomentations import Compose, AddColoredNoise, PitchShift, Gain

/home/zakhar/miniconda3/envs/aphasia_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1787331002.674804   99102 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787331002.722767   99102 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787331004.022435   99102 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to 

In [2]:
SEED = 1984

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

gen = set_seed(SEED)

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AUDIO_LENGTH = 6_000
SEQUENCE_LENGTH = 420
MFCC = 128
print(f"It's {DEVICE} time!!!")

It's cuda time!!!


In [4]:
DATA_DIR = os.path.join(os.getcwd(), 'data')
VOICES_DIR = os.path.join(DATA_DIR, 'Voices')
APHASIA_DIR = os.path.join(VOICES_DIR, 'Aphasia')
NORM_DIR = os.path.join(VOICES_DIR, 'Norm')

In [5]:
train_dataset = AphasiaDatasetMFCC(os.path.join(DATA_DIR, "train_filenames.csv"), VOICES_DIR, target_sample_rate=8_000, mfcc=MFCC, n_mels=128, fft_size=512,
                 hop_length=256, win_length=512, min_duration=10, max_duration=15, seed=SEED)
val_dataset = AphasiaDatasetMFCC(os.path.join(DATA_DIR, "val_filenames.csv"), VOICES_DIR, target_sample_rate=8_000, mfcc=MFCC, n_mels=128, fft_size=512,
                 hop_length=256, win_length=512, min_duration=10, max_duration=15, seed=SEED)
test_dataset = AphasiaDatasetMFCC(os.path.join(DATA_DIR, "test_filenames.csv"), VOICES_DIR, target_sample_rate=8_000, mfcc=MFCC, n_mels=128, fft_size=512,
                 hop_length=256, win_length=512, min_duration=10, max_duration=15, seed=SEED)

# Балансировка классов для train
train_labels = [label for _, label in train_dataset.data]
class_counts = Counter(train_labels)
if len(class_counts) < 2:
    raise ValueError("Один из классов отсутствует в тренировочном наборе")

class_weights = {label: 1.0 / count for label, count in class_counts.items()}
weights = [class_weights[label] for _, label in train_dataset.data]
gen = set_seed(SEED)
train_sampler = WeightedRandomSampler(weights, num_samples=len(train_dataset), replacement=True, generator=gen)

In [6]:
def pad_sequence(batch):
    if not batch:
        return torch.zeros(0), torch.zeros(0)
    
    seq, labels = zip(*batch)
    max_len = max(s.shape[1] for s in seq)

    padded = torch.zeros(len(seq), MFCC, SEQUENCE_LENGTH)
    for i, s in enumerate(seq):
        padded[i, ..., :s.shape[-1]] = s[..., :SEQUENCE_LENGTH]
    
    return padded, torch.stack(labels) 

In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=16, sampler=train_sampler, collate_fn=pad_sequence, drop_last=True, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=pad_sequence, drop_last=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=pad_sequence, drop_last=True, num_workers=0)

In [8]:
def train_model(model, dl_train, dl_val, epochs=1, lr=0.001, device="cpu", path="checkpoints", model_name="model", seed=SEED):
    set_seed(seed)
    if getattr(dl_train, "sampler", None) is not None:
        sampler_gen = torch.Generator()
        sampler_gen.manual_seed(seed)
        dl_train.sampler.generator = sampler_gen

    model = model.to(device)
    criterion = nn.CrossEntropyLoss().to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3, threshold=1e-3)
        
    train_loss_list = []
    val_loss_list = []
    train_acc_list = []
    val_acc_list = []
    max_val_acc = 0
    for epoch in tqdm(range(epochs), desc="Training model"):
        model.train()
        total_train_loss = 0
        total_val_loss = 0
        
        train_acc = []
        val_acc = []
        for features, target in dl_train:
            features, target = features.to(device), target.to(device)
            optimizer.zero_grad()
            
            output = model(features.unsqueeze(1)).squeeze()

            preds = torch.argmax(output, dim=1).cpu().detach().numpy()
            train_acc.append(accuracy_score(target.cpu().detach().numpy(), preds))
            
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.detach().item()
        
        avg_train_acc = np.stack(train_acc, axis=0).mean()

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_loss_list.append(avg_train_loss)
        train_acc_list.append(avg_train_acc)
                
        model.eval()
        
        with torch.no_grad():
            for features, target in dl_val:
                features, target = features.to(device), target.to(device)
                
                output = model(features.unsqueeze(1)).squeeze()
                
                preds = torch.argmax(output, dim=1).cpu().detach().numpy()
                val_acc.append(accuracy_score(target.cpu().detach().numpy(), preds))
                loss = criterion(output, target)
                total_val_loss += loss.detach().item()
        
        avg_val_acc = np.stack(val_acc, axis=0).mean()
        avg_val_loss = total_val_loss / len(dl_val)
        val_loss_list.append(avg_val_loss)
        val_acc_list.append(avg_val_acc)
        
        if scheduler:
            try:
                scheduler.step()
            except:
                scheduler.step(avg_val_loss)
        
        if max_val_acc <= avg_val_acc:
            if not os.path.exists(path):
                os.makedirs(path)
                
            chkp = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                "plots": {
                    "train_loss": train_loss_list,
                    "train_acc": train_acc_list,
                    "val_loss": val_loss_list,
                    "val_acc": val_acc_list
                },
            }
            torch.save(chkp, os.path.join(path, model_name + ".pt"))
            max_val_acc = avg_val_acc

        # if epoch % 10 == 0:
        tqdm.write(f"Epoch {epoch}: train loss: {avg_train_loss:.3f}, train balanced acc: {avg_train_acc:.2f}, test loss: {avg_val_loss:.3f}, test balanced acc: {avg_val_acc:.2f}, lr: {optimizer.param_groups[0]['lr']}")

            
    return model, train_loss_list, val_loss_list, train_acc_list, val_acc_list

In [9]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=2, in_channels=1):
        super(CNNModel, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),  # Conv 1
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)), # Conv 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)), # Conv 3
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)), # Conv 4
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)), # Conv 5
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
        )

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))  # Усреднение перед FC-слоями

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.global_avg_pool(x)
        x = self.fc_layers(x)
        return x

In [10]:
set_seed(SEED)
cnn = CNNModel(num_classes=2, in_channels=1)

In [11]:
cnn, train_l, val_l, train_accuracy, val_accuracy = train_model(cnn, train_dataloader, val_dataloader, epochs=10, lr=1e-3, 
                                                                device=DEVICE, path=os.path.join("checkpoints", "cnn"), model_name="cnn_model_mfcc_2d")

Training model:  10%|█         | 1/10 [00:27<04:11, 27.90s/it]

Epoch 0: train loss: 0.580, train balanced acc: 0.70, test loss: 0.570, test balanced acc: 0.73, lr: 0.001


Training model:  20%|██        | 2/10 [00:55<03:42, 27.77s/it]

Epoch 1: train loss: 0.520, train balanced acc: 0.74, test loss: 0.449, test balanced acc: 0.81, lr: 0.001


Training model:  30%|███       | 3/10 [01:23<03:14, 27.74s/it]

...

Epoch 7: train loss: 0.344, train balanced acc: 0.85, test loss: 0.588, test balanced acc: 0.84, lr: 0.001


Training model:  90%|█████████ | 9/10 [04:11<00:28, 28.19s/it]

Epoch 8: train loss: 0.318, train balanced acc: 0.87, test loss: 1.046, test balanced acc: 0.60, lr: 0.001


Training model: 100%|██████████| 10/10 [04:40<00:00, 28.05s/it]

Epoch 9: train loss: 0.314, train balanced acc: 0.87, test loss: 0.457, test balanced acc: 0.84, lr: 0.001


In [12]:
state_d = torch.load(os.path.join("checkpoints", "cnn", "cnn_model_mfcc_2d.pt"), weights_only=False)

cnn = CNNModel(num_classes=2, in_channels=1)
cnn.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [13]:
def test_model(model, test_data):
    model = model.to(DEVICE)
        
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for features, target in tqdm(test_data):
            features = features.to(DEVICE)
            target = target.to(DEVICE)
            
            label = model(features.unsqueeze(1)).to("cpu").detach().numpy().squeeze()
            preds.extend(label.argmax(axis=-1).tolist())
            # print(target.shape)
            targets.extend(target.to("cpu").tolist())

    preds = np.array(preds)
    # print(targets)
    print(classification_report(targets, preds))    

    return preds

In [14]:
test_model(cnn, test_dataloader)

100%|██████████| 59/59 [00:03<00:00, 18.60it/s]

              precision    recall  f1-score   support

           0       0.70      0.83      0.76       122
           1       0.97      0.95      0.96       822

    accuracy                           0.93       944
   macro avg       0.84      0.89      0.86       944
weighted avg       0.94      0.93      0.93       944



array([1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0,
       0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [15]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 20))

# axes[0][0].plot(train_l)
# axes[0][0].set_title("Train loss", fontsize=30)
# axes[0][0].set_xlabel("Epoch", fontsize=15)

# axes[0][1].plot(val_l)
# axes[0][1].set_title("Validation loss", fontsize=30)
# axes[0][1].set_xlabel("Epoch", fontsize=15)

# axes[1][0].plot(train_accuracy)
# axes[1][0].set_title("Train accuracy", fontsize=30)
# axes[1][0].set_xlabel("Epoch", fontsize=15)

# axes[1][1].plot(val_accuracy)
# axes[1][1].set_title("Validation accuracy", fontsize=30)
# axes[1][1].set_xlabel("Epoch", fontsize=15)

In [16]:
test_data = pd.read_csv(os.path.join(DATA_DIR, 'test_filenames.csv'))
DATA_DIR = os.path.join(os.getcwd(), 'data')
VOICES_DIR = os.path.join(DATA_DIR, 'Voices_wav')
APHASIA_DIR = os.path.join(VOICES_DIR, 'Aphasia')
NORM_DIR = os.path.join(VOICES_DIR, 'Norm')

In [17]:
def test_model_for_each_participant(model, test_data):
    model = model.to("cpu")
        
    model.eval()

    test_data["ID"] = test_data["file_name"].apply(
        lambda x: str(x).split("-")[0] + str(x).split("-")[1])
    test_data.head()
    IDs = test_data["ID"].unique()

    all_preds = []
    all_labels = []
    
    errors = []
    
    with torch.no_grad():
        for participant_id in tqdm(IDs):
            participant_samples = test_data[test_data["ID"] == participant_id]
            preds = []
            for ind, participant_sample in participant_samples.iterrows():

                sgnl_path = participant_sample["file_name"]

                if sgnl_path[0] == "N":
                    sgnl_path = os.path.join(NORM_DIR, sgnl_path)
                else:
                    sgnl_path = os.path.join(APHASIA_DIR, sgnl_path)
                    

                chunks = train_dataset.process_audio(sgnl_path)

                padded = torch.zeros(len(chunks), MFCC, SEQUENCE_LENGTH)
                for i, s in enumerate(chunks):

                    padded[i, ..., :s.shape[-1]] = s[..., :SEQUENCE_LENGTH]
                # print(model(torch.from_numpy(np.array(padded)).unsqueeze(1)).detach().numpy().shape)
                pred = model(torch.from_numpy(np.array(padded)).unsqueeze(1)).detach().numpy()# .squeeze(1)# .argmax(axis=-1)
                preds.append(pred)
            labels = participant_samples["label"]
            # print(preds)
            pred = np.concatenate(preds, axis=0).mean(axis=-2).argmax(axis=-1)

            all_preds.append(pred)
                
            all_labels.append(labels.values[0])
            errs = participant_samples[labels.values != pred]
            if len(errs) > 0:
                errors.append(errs)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    print(classification_report(all_labels, all_preds))

    
    return all_preds, errors

In [18]:
test_model_for_each_participant(cnn, test_data)

  0%|          | 0/72 [00:00<?, ?it/s]/tmp/ipykernel_99102/3300300607.py:37: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pred = model(torch.from_numpy(np.array(padded)).unsqueeze(1)).detach().numpy()# .squeeze(1)# .argmax(axis=-1)
100%|██████████| 72/72 [04:23<00:00,  3.66s/it]

              precision    recall  f1-score   support

           0       0.95      0.91      0.93        23
           1       0.96      0.98      0.97        49

    accuracy                           0.96        72
   macro avg       0.96      0.95      0.95        72
weighted avg       0.96      0.96      0.96        72



(array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 0, 1, 1]),
 [                file_name  label     ID
  22  N-1005-RAT-1-bike.wav      0  N1005
  23  N-1005-RAT-1-robb.wav      0  N1005,
                  file_name  label     ID
  30  A-1237-RAT-1-bike.wav      0  A1237
  31  A-1237-RAT-1-robb.wav      0  A1237,
                  file_name  label    ID
  160  A-215-RAT-1-robb.wav      1  A215
  161  A-215-RAT-1-bike.wav      1  A215])

In [19]:
import torchvision.models as models

class MobileNet(nn.Module):
    
    def __init__(self, num_classes=2,):
        super(MobileNet, self).__init__()
        self.features = models.mobilenet_v3_large(pretrained=False)
        
        for child in self.features.children():
            for param in child.parameters():
                param.requires_grad = True
        
        in_features = self.features.classifier[0].in_features
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
        self.features.classifier = self.classifier
    
    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)

        x = self.features(x)
        return x
        

In [20]:
set_seed(SEED)
mobilenet = MobileNet()

In [21]:
mobilenet, train_l, val_l, train_accuracy, val_accuracy = train_model(mobilenet, train_dataloader, val_dataloader, epochs=10, lr=1e-3, 
                                                                      device=DEVICE, path=os.path.join("checkpoints", "mobilenet"), model_name="mobilenet_mfcc_2d")

Training model:  10%|█         | 1/10 [00:05<00:45,  5.05s/it]

Epoch 0: train loss: 0.498, train balanced acc: 0.77, test loss: 0.690, test balanced acc: 0.79, lr: 0.001


Training model:  20%|██        | 2/10 [00:10<00:40,  5.02s/it]

Epoch 1: train loss: 0.357, train balanced acc: 0.86, test loss: 0.629, test balanced acc: 0.79, lr: 0.001


Training model:  30%|███       | 3/10 [00:14<00:34,  4.99s/it]

...

Epoch 7: train loss: 0.066, train balanced acc: 0.98, test loss: 0.934, test balanced acc: 0.84, lr: 0.0005


Training model:  90%|█████████ | 9/10 [00:44<00:04,  4.93s/it]

Epoch 8: train loss: 0.063, train balanced acc: 0.98, test loss: 1.091, test balanced acc: 0.72, lr: 0.0005


Training model: 100%|██████████| 10/10 [00:49<00:00,  4.95s/it]

Epoch 9: train loss: 0.036, train balanced acc: 0.99, test loss: 1.198, test balanced acc: 0.74, lr: 0.0005


In [22]:
model_sd = torch.load(os.path.join("checkpoints", "mobilenet", "mobilenet_mfcc_2d.pt"), weights_only=False)

mobilenet = MobileNet()
mobilenet.load_state_dict(model_sd["model_state_dict"])

<All keys matched successfully>

In [23]:
_ = test_model(mobilenet, test_dataloader)

  0%|          | 0/59 [00:00<?, ?it/s]

100%|██████████| 59/59 [00:00<00:00, 123.07it/s]

              precision    recall  f1-score   support

           0       0.57      0.71      0.63       122
           1       0.96      0.92      0.94       822

    accuracy                           0.89       944
   macro avg       0.76      0.82      0.79       944
weighted avg       0.91      0.89      0.90       944



In [24]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 20))

# axes[0][0].plot(train_l)
# axes[0][0].set_title("Train loss", fontsize=30)
# axes[0][0].set_xlabel("Epoch", fontsize=15)

# axes[0][1].plot(val_l)
# axes[0][1].set_title("Validation loss", fontsize=30)
# axes[0][1].set_xlabel("Epoch", fontsize=15)

# axes[1][0].plot(train_accuracy)
# axes[1][0].set_title("Train accuracy", fontsize=30)
# axes[1][0].set_xlabel("Epoch", fontsize=15)

# axes[1][1].plot(val_accuracy)
# axes[1][1].set_title("Validation accuracy", fontsize=30)
# axes[1][1].set_xlabel("Epoch", fontsize=15)

In [25]:
preds_arr, incorrect_samples = test_model_for_each_participant(mobilenet, test_data)

  0%|          | 0/72 [00:00<?, ?it/s]/tmp/ipykernel_99102/3300300607.py:37: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pred = model(torch.from_numpy(np.array(padded)).unsqueeze(1)).detach().numpy()# .squeeze(1)# .argmax(axis=-1)
100%|██████████| 72/72 [00:20<00:00,  3.59it/s]

              precision    recall  f1-score   support

           0       0.84      0.70      0.76        23
           1       0.87      0.94      0.90        49

    accuracy                           0.86        72
   macro avg       0.86      0.82      0.83        72
weighted avg       0.86      0.86      0.86        72



In [26]:
from pydub import AudioSegment

def test_model_no_chunks(model, test_data):
    model = model.to("cpu")
        
    model.eval()

    test_data["ID"] = test_data["file_name"].apply(
        lambda x: str(x).split("-")[0] + str(x).split("-")[1])
    test_data.head()
    IDs = test_data["ID"].unique()

    all_preds_aggr = []
    all_labels_aggr = []
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for participant_id in tqdm(IDs):
            participant_samples = test_data[test_data["ID"] == participant_id]
            preds_aggr = []
            labels_aggr = []
            for ind, participant_sample in participant_samples.iterrows():

                sgnl_path = participant_sample["file_name"]

                if sgnl_path[0] == "N":
                    sgnl_path = os.path.join(NORM_DIR, sgnl_path)
                else:
                    sgnl_path = os.path.join(APHASIA_DIR, sgnl_path)
                    
                # y, sr = librosa.load(sgnl_path, sr=8_000)
                audio = AudioSegment.from_file(sgnl_path, format="3gp")
                y = train_dataset.preprocess(audio)

                pred = model(y[None, None, ...]).detach().numpy().squeeze().argmax(axis=-1)
                
                labels_aggr.append(participant_sample['label'])
                preds_aggr.append(pred)

            pred = scipy.stats.mode(np.array(preds_aggr))

            all_preds_aggr.append(pred.mode)
            all_labels_aggr.append(labels_aggr[0])
            
            all_preds.extend(preds_aggr)
            all_labels.extend(labels_aggr)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    print("Without aggr")
    print(classification_report(all_labels, all_preds))
    
    print("With aggr")
    print(classification_report(all_labels_aggr, all_preds_aggr))
    
    return all_preds, all_preds_aggr

In [27]:
_, _ = test_model_no_chunks(mobilenet, test_data)

100%|██████████| 72/72 [00:14<00:00,  4.81it/s]

Without aggr
              precision    recall  f1-score   support

           0       0.83      0.76      0.80        46
           1       0.91      0.94      0.93       122

    accuracy                           0.89       168
   macro avg       0.87      0.85      0.86       168
weighted avg       0.89      0.89      0.89       168

With aggr
              precision    recall  f1-score   support

           0       0.83      0.87      0.85        23
           1       0.94      0.92      0.93        49

    accuracy                           0.90        72
   macro avg       0.89      0.89      0.89        72
weighted avg       0.90      0.90      0.90        72



### False Negative and False Positive samples

In [28]:
incorrect_samples

[               file_name  label     ID
 0  N-0916-RAT-1-bike.wav      0  N0916
 1  N-0916-RAT-1-robb.wav      0  N0916,
                file_name  label     ID
 6  N-1016-RAT-1-robb.wav      0  N1016
 7  N-1016-RAT-1-bike.wav      0  N1016,
                 file_name  label     ID
 16  N-0940-RAT-1-robb.wav      0  N0940
 17  N-0940-RAT-1-bike.wav      0  N0940,
                 file_name  label     ID
 22  N-1005-RAT-1-bike.wav      0  N1005
 23  N-1005-RAT-1-robb.wav      0  N1005,
                 file_name  label     ID
 26  N-0912-RAT-1-robb.wav      0  N0912
 27  N-0912-RAT-1-bike.wav      0  N0912,
                 file_name  label     ID
 30  A-1237-RAT-1-bike.wav      0  A1237
 31  A-1237-RAT-1-robb.wav      0  A1237,
                 file_name  label     ID
 42  N-0923-RAT-1-bike.wav      0  N0923
 43  N-0923-RAT-1-robb.wav      0  N0923,
                file_name  label    ID
 58  A-932-RAT-1-robb.wav      1  A932
 59  A-932-RAT-1-bike.wav      1  A932,
                file

In [29]:
simple_features = np.load(os.path.join(DATA_DIR, 'test_data_simple.npy'))
test_filenames = pd.read_csv(os.path.join(DATA_DIR, 'test_filenames.csv'))

In [30]:
error_idxes = [y for x in incorrect_samples for y in x.index]

In [31]:
simple_features[np.array(error_idxes)]

array([[5.27808000e+05, 2.00000000e+00, 2.63904000e+05, 1.87219200e+06,
        3.00000000e+00, 6.24064000e+05, 2.81919803e-01],
       [2.21408000e+05, 1.00000000e+00, 2.21408000e+05, 2.17859200e+06,
        2.00000000e+00, 1.08929600e+06, 1.01628942e-01],
       [1.16704000e+05, 1.00000000e+00, 1.16704000e+05, 2.28329600e+06,
        2.00000000e+00, 1.14164800e+06, 5.11120766e-02],
       [1.40960000e+05, 3.00000000e+00, 4.69866667e+04, 2.25904000e+06,
        4.00000000e+00, 5.64760000e+05, 6.23981868e-02],
       [7.69920000e+04, 2.00000000e+00, 3.84960000e+04, 2.32300800e+06,
        3.00000000e+00, 7.74336000e+05, 3.31432350e-02],
       [1.22048000e+05, 2.00000000e+00, 6.10240000e+04, 2.27795200e+06,
        3.00000000e+00, 7.59317333e+05, 5.35779507e-02],
       [3.00800000e+05, 8.00000000e+00, 3.76000000e+04, 2.09920000e+06,
        9.00000000e+00, 2.33244444e+05, 1.43292683e-01],
       [2.45856000e+05, 5.00000000e+00, 4.91712000e+04, 2.15414400e+06,
        6.00000000e+00, 3

In [32]:
# CHECKPOINTS_DIR = os.path.join("..", "checkpoints")
# if not os.path.exists(os.path.join(CHECKPOINTS_DIR, "mobilenet_chkp")):
#     os.makedirs(os.path.join(CHECKPOINTS_DIR, "mobilenet_chkp"))

# torch.save(
#     {
#         'epoch': 10,
#         'model_state_dict': mobilenet.state_dict(),
#         # 'optimizer_state_dict': optimizer.state_dict(),
#         # 'scheduler_state_dict': scheduler.state_dict(),
#         'losses_train': train_l,
#         'losses_val': val_l,
#     },
#     os.path.join(CHECKPOINTS_DIR, "mobilenet_chkp", "mobilenet" + "_" + f"{val_accuracy[-1]:.2f}" + '.pt'),)